# FER-CE Baseline: ResNet50 for Emotion Recognition

This notebook implements a baseline model for the Recognition of Affective Facial Expressions (RAF-CE) dataset using a pre-trained ResNet50 architecture.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Add src to path
sys.path.append(os.path.abspath('../src'))
from dataset import RAFCEDataset, get_transforms

## 1. Parameters & Hyperparameters

In [ ]:
IMG_DIR = '../../aligned'
LABEL_FILE = '../../RAFCE_emolabel.txt'
PARTITION_FILE = '../../RAFCE_partition.txt'
OUTPUT_DIR = '../outputs/baseline'
BATCH_SIZE = 32
EPOCHS = 20
LR = 0.001
NUM_CLASSES = 14
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Data Loading

In [ ]:
train_transform = get_transforms(augment=True)
val_transform = get_transforms(augment=False)

# Note: Split IDs (0=train, 1=val, 2=test) are based on RAF-CE usual splits.
# If your partition file uses different IDs, update here.
train_dataset = RAFCEDataset(IMG_DIR, LABEL_FILE, PARTITION_FILE, split=1, transform=train_transform) # assuming 1 is train in this set
test_dataset = RAFCEDataset(IMG_DIR, LABEL_FILE, PARTITION_FILE, split=2, transform=val_transform) # assuming 2 is test

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 3. Model Definition

In [ ]:
model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

## 4. Training Loop

In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs):
    train_losses = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")
    return train_losses

losses = train_model(model, train_loader, criterion, optimizer, EPOCHS)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'resnet50_ferce.pth'))

## 5. Evaluation

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    print(classification_report(all_labels, all_preds))
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'))
    plt.show()

evaluate(model, test_loader)